# example.ipynb

<div align="center">
  <a href="https://elliot-ai.eu"><img src="https://data.source.coop/major-tom/index/images/elliot.png" height="80"></a>
  &nbsp;&nbsp;&nbsp;&nbsp;
  <a href="https://asterisk.coop"><img src="https://data.source.coop/major-tom/index/images/asterisk.png" height="80"></a>
  &nbsp;&nbsp;&nbsp;&nbsp;
  <a href="https://isp.uv.es"><img src="https://data.source.coop/major-tom/index/images/isp.png" height="80"></a>
</div>

# **ELLIOT-Pretrain**

[![Source Cooperative](https://img.shields.io/badge/Source_Cooperative-elliot--pretrain-blue)](https://source.coop/major-tom/elliot-pretrain)
[![TACO v3](https://img.shields.io/badge/format-TACO_v3-green)](https://tacofoundation.github.io)
[![License](https://img.shields.io/badge/license-CC--BY--SA--4.0%20%2B%20ODbL-orange)](https://creativecommons.org/licenses/by-sa/4.0/)

[ELLIOT-Pretrain](https://source.coop/major-tom/elliot-pretrain) is a multi-modal Earth Observation dataset designed for foundation model pretraining. It is built on the [MajorTOM](https://github.com/ESA-PhiLab/Major-TOM) global 10 km grid and comes in three temporal regimes:

| Part | Name | Tiles | Timesteps | Cadence | Purpose |
|------|------|-------|-----------|---------|---------|
| **A** | Monotemporal | 250,000 | 1 | — | Global coverage, spatial diversity |
| **B** | Monthly | 12,500 | 12 | one per calendar month, across years | Seasonal dynamics, phenology |
| **C** | Burst | 16,666 | 6 | ~5-day | Rapid events: floods, fires, landslides |

Every tile carries the same layers, co-registered on one 10 m grid:

| Layer | Source | Resolution | Content |
|--------|--------|------------|-------|
| Sentinel-2 L1C | ESA Copernicus | 10 m | 13 spectral bands |
| Landsat 8/9 OLI-TIRS | USGS | 30 m | 11 bands (9 OLI + 2 TIRS) |
| Copernicus DEM GLO-30 | ESA / TanDEM-X | 30 m | elevation |
| ESA WorldCover | ESA | 10 m | land cover |
| Sentinel-1 RTC | Catalyst, via Microsoft Planetary Computer | 10 m | VV, VH, one image per Sentinel-2 frame |
| Cloud and shadow masks | OmniCloudMask | 10 m | one per Sentinel-2 and Landsat frame |
| ERA5 | Copernicus Climate Change Service | — | weather at each acquisition |
| OpenStreetMap | © OpenStreetMap contributors | vector | the tile's features and administrative units |

Tile locations were selected via hierarchical spherical k-means over [AlphaEarth Foundation](https://huggingface.co/Major-TOM/AlphaEarth-Foundations) embeddings to maximise global environmental diversity. The dataset is packaged as [TACO v3](https://tacofoundation.github.io) folders: no special libraries are needed, just `pandas`, `rasterio` and `matplotlib`. Sentinel-1, the cloud masks, ERA5 and OpenStreetMap were added by the Image and Signal Processing Group (ISP), Universitat de València.

Licence: CC-BY-SA-4.0, except `osm.parquet` and `admin.parquet`, which are OpenStreetMap extracts under the Open Database License (ODbL-1.0).

## 1. Setup

In [ ]:
%pip install -q pandas pyarrow geopandas rasterio matplotlib requests pillow

In [ ]:
import io
import os

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from PIL import Image

BASE_URL = os.environ.get("ELLIOT_BASE_URL", "https://data.source.coop/major-tom/elliot-pretrain")
HEADERS = {"User-Agent": "Mozilla/5.0"}


def get_json(path):
    return requests.get(f"{BASE_URL}/{path}", headers=HEADERS).json()


def get_parquet(path):
    """A parquet file of the release, as a DataFrame."""
    with requests.get(f"{BASE_URL}/{path}", headers=HEADERS) as r:
        return pd.read_parquet(io.BytesIO(r.content))


def get_geoparquet(path):
    with requests.get(f"{BASE_URL}/{path}", headers=HEADERS) as r:
        return gpd.read_parquet(io.BytesIO(r.content))


def frame_rows(df, idx):
    """The rows of one tile in a per-file metadata table, in file order.

    Every row's `internal:relative_path` starts with the tile's own folder, `<idx>/`.
    """
    rows = df[df["internal:relative_path"].str.startswith(f"{idx}/")]
    return rows.sort_values("internal:relative_path")


def rgb(src, bands=(4, 3, 2), scale=3000):
    return np.clip(src.read(list(bands)).astype(np.float32) / scale, 0, 1).transpose(1, 2, 0)

The collection is organised as three TACO v3 datasets under a shared root:

```
elliot-pretrain/
├── monotemporal/          # Part A — 250k tiles, 1 timestep
│   ├── COLLECTION.json
│   ├── METADATA/
│   │   ├── sample.parquet             one row per tile
│   │   ├── children.parquet           one row per top-level entry of a tile
│   │   ├── children__s2.parquet       one row per Sentinel-2 file (+ ERA5)
│   │   ├── children__l8.parquet       one row per Landsat file (+ ERA5)
│   │   ├── children__s1.parquet       one row per Sentinel-1 file
│   │   ├── children__cloud_s2.parquet one row per Sentinel-2 cloud mask
│   │   ├── children__cloud_l8.parquet one row per Landsat cloud mask
│   │   ├── children__dem.parquet
│   │   └── children__lc.parquet
│   └── DATA/{idx}/
│       ├── s2/data.tif   l8/data.tif   dem/data.tif   lc/data.tif
│       ├── s1/data.tif   cloud_s2/data.png   cloud_l8/data.png
│       ├── osm.parquet
│       └── admin.parquet
├── monthly/               # Part B — 12.5k tiles, 12 timesteps
│   └── DATA/{idx}/
│       ├── s2/img_00.tif .. img_11.tif     l8/img_00.tif .. img_11.tif
│       ├── s1/img_00.tif .. img_11.tif     cloud_s2/img_00.png ..   cloud_l8/img_00.png ..
│       ├── dem/main.tif   lc/main.tif
│       └── osm.parquet   admin.parquet
└── burst/                 # Part C — 16.7k tiles, 6 timesteps (~5-day)
    └── DATA/{idx}/        same as monthly, img_00 .. img_05
```

## 2. Part A — Monotemporal

**250,000 globally distributed tiles**, each with one best cloud-free scene per sensor. Designed for tasks where spatial coverage matters: land cover classification, feature extraction, and foundation model pretraining.

### 2a. Contract

`COLLECTION.json` describes the structure, every metadata column, and an `ml:contract` declaring each layer's bands, units and scaling.

In [ ]:
col_a = get_json("monotemporal/COLLECTION.json")
print(f"ID:        {col_a['id']}")
print(f"Licences:  {col_a['licenses']}")
print(f"Structure: {col_a['taco:structure']}")
print("\nMetadata levels:")
for key in col_a["taco:metadata"]:
    fields = list(col_a["taco:metadata"][key].keys())
    print(f"  {key}: {len(fields)} fields → {fields[:4]}...")
print("\nLayers:")
for slot in col_a["ml:contract"]["inputs"]:
    print(f"  {slot['name']:14s} {slot['kind']:14s} scale {slot.get('scale_factor')}")

### 2b. Metadata

`sample.parquet` has one row per tile with shared context: location, climate, soil, socioeconomic, admin boundaries, and per-tile summaries of the added layers. The tile centroid is `ml:geometry`, WKB in EPSG:4326.

In [ ]:
df_a = get_parquet("monotemporal/METADATA/sample.parquet")
df_a = gpd.GeoDataFrame(df_a, geometry=gpd.GeoSeries.from_wkb(df_a["ml:geometry"]), crs=4326)

spain = df_a[(df_a["admin:country"] == "Spain") & (df_a["terrain:elevation"] > 1000)]
print(f"Tiles in Spain above 1000m: {len(spain):,}")
spain[["majortom:code_10km", "terrain:elevation", "admin:state", "cloud:s2_mean_fraction"]].head()

### 2c. Imagery

Each tile's layers are files under `DATA/{idx}/`. `rasterio` reads them directly as remote COGs via GDAL `/vsicurl/`, no download needed.

In [ ]:
IDX = 140510
MODS = ["s2", "l8", "dem", "lc"]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for j, mod in enumerate(MODS):
    url = f"/vsicurl/{BASE_URL}/monotemporal/DATA/{IDX}/{mod}/data.tif"
    with rasterio.open(url) as src:
        if mod in ("s2", "l8"):
            axes[j].imshow(rgb(src))
        elif mod == "dem":
            axes[j].imshow(src.read(1), cmap="terrain")
        else:
            axes[j].imshow(src.read(1), cmap="tab20")
    axes[j].set_title(mod.upper())
    axes[j].axis("off")

plt.suptitle(f"Monotemporal — {df_a['admin:state'].iloc[IDX]}, {df_a['admin:country'].iloc[IDX]}", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Part B, Monthly

**12,500 tiles across 12 timesteps**. One S2 and one L8 scene per calendar month, selected with greedy circular DOY spacing over the historical archive: the 12 least cloudy scenes that are maximally spread across the year. The months need not come from the same year, so a series is a multi-year climatology. DEM and WorldCover are static (one raster per tile).

### 3.1 Contract

In [ ]:
col_b = get_json("monthly/COLLECTION.json")
print(f"ID        {col_b['id']}")
print(f"Structure {col_b['taco:structure'][:4]} ... {len(col_b['taco:structure'])} entries")
print("\nMetadata levels")
for key in col_b["taco:metadata"]:
    fields = list(col_b["taco:metadata"][key].keys())
    print(f"  {key}, {len(fields)} fields, {fields[:4]}...")

### 3.2 Metadata

One parquet file per level under `METADATA/`:

- `sample.parquet`, one row per tile (12,500 rows). Shared context, same schema as A.
- `children__s2.parquet`, 12 rows per tile. Acquisition time, product, angles, and ERA5 weather at the acquisition.
- `children__l8.parquet`, 12 rows per tile, the same for Landsat 8/9.
- `children__cloud_s2.parquet` and `children__cloud_l8.parquet`, one row per cloud mask with its cloud, thin-cloud and shadow fractions.
- `children__s1.parquet`, one row per Sentinel-1 image: the S2 frame it pairs with, its own time, orbit and valid fraction.
- `children__dem.parquet` and `children__lc.parquet`, one row per static raster.

A row belongs to the tile whose index starts its `internal:relative_path` (`400/s2/img_03.tif`).

In [ ]:
df_b = get_parquet("monthly/METADATA/sample.parquet")
df_s2 = get_parquet("monthly/METADATA/children__s2.parquet")
df_l8 = get_parquet("monthly/METADATA/children__l8.parquet")
df_cloud = get_parquet("monthly/METADATA/children__cloud_s2.parquet")

for name, df in [("sample", df_b), ("children__s2", df_s2), ("children__l8", df_l8),
                 ("children__cloud_s2", df_cloud)]:
    print(f"{name:20s} {len(df):>8,}")

Show the 12 `S2` and 12 `L8` rows for a single tile, with the cloud fraction of each S2 frame.

In [ ]:
tile_idx = 400
tile_code = df_b["majortom:code_10km"].iloc[tile_idx]

s2_rows = frame_rows(df_s2, tile_idx).assign(
    datetime=lambda d: pd.to_datetime(d["ml:time_start"], unit="ms"),
    cloud=frame_rows(df_cloud, tile_idx)["ml:cloud_fraction"].to_numpy())
l8_rows = frame_rows(df_l8, tile_idx)

print(f"Tile {tile_idx}  code {tile_code}\n")
print("S2")
print(s2_rows[["internal:relative_path", "datetime", "cloud", "ml:era5_temperature_2m_c"]].to_string(index=False))
print("\nL8")
print(l8_rows[["internal:relative_path", "l8:date", "l8:doy", "l8:cloud_cover", "l8:wrs_path", "l8:wrs_row"]].to_string(index=False))

### 3.3 Imagery

S2 and L8 have 12 timesteps each (`img_00.tif` to `img_11.tif`). DEM and LC are static (`main.tif`).

In [ ]:
tile_idx = 8000

fig, axes = plt.subplots(2, 6, figsize=(24, 8))
for i, ax in enumerate(axes.flat):
    with rasterio.open(f"/vsicurl/{BASE_URL}/monthly/DATA/{tile_idx}/s2/img_{i:02d}.tif") as src:
        ax.imshow(rgb(src))
    ax.set_title(f"img_{i:02d}", fontsize=10)
    ax.axis("off")
plt.suptitle(f"S2, tile {tile_idx}", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 6, figsize=(24, 8))
for i, ax in enumerate(axes.flat):
    with rasterio.open(f"/vsicurl/{BASE_URL}/monthly/DATA/{tile_idx}/l8/img_{i:02d}.tif") as src:
        ax.imshow(rgb(src))
    ax.set_title(f"img_{i:02d}", fontsize=10)
    ax.axis("off")
plt.suptitle(f"L8, tile {tile_idx}", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for j, (mod, cmap) in enumerate([("dem", "terrain"), ("lc", "tab20")]):
    with rasterio.open(f"/vsicurl/{BASE_URL}/monthly/DATA/{tile_idx}/{mod}/main.tif") as src:
        axes[j].imshow(src.read(1), cmap=cmap)
    axes[j].set_title(mod.upper())
    axes[j].axis("off")
plt.suptitle(f"Static layers, tile {tile_idx}", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Part C, Burst

**16,666 tiles across 6 timesteps**. Six S2 and six L8 scenes within a 30-day window, giving around 5-day cadence. The 30-day window is chosen automatically per tile as the best run of consecutive days in the 2017 to 2025 archive, where "best" means low cloud cover and dense revisit. Designed for rapid event detection such as floods, fires, landslides, and sub-monthly change detection. DEM and WorldCover are static, same as in B.

### 4.1 Contract

In [ ]:
col_c = get_json("burst/COLLECTION.json")
print(f"ID        {col_c['id']}")
print(f"Structure {col_c['taco:structure'][:4]} ... {len(col_c['taco:structure'])} entries")

### 4.2 Metadata

Same layout as B. Pull the 6 `S2` and 6 `L8` rows of one tile and inspect the burst window.

In [ ]:
df_c = get_parquet("burst/METADATA/sample.parquet")
df_c_s2 = get_parquet("burst/METADATA/children__s2.parquet")
df_c_l8 = get_parquet("burst/METADATA/children__l8.parquet")

tile_idx = 1000
s2_rows = frame_rows(df_c_s2, tile_idx).assign(datetime=lambda d: pd.to_datetime(d["ml:time_start"], unit="ms"))
l8_rows = frame_rows(df_c_l8, tile_idx).assign(datetime=lambda d: pd.to_datetime(d["l8:date"]))

all_dt = pd.concat([s2_rows["datetime"], l8_rows["datetime"]])
print(f"Tile {tile_idx} ({df_c['majortom:code_10km'].iloc[tile_idx]}), "
      f"from {all_dt.min().date()} to {all_dt.max().date()}, {(all_dt.max() - all_dt.min()).days} days\n")
print("S2")
print(s2_rows[["internal:relative_path", "datetime", "s2:nodata_pct"]].to_string(index=False))
print("\nL8")
print(l8_rows[["internal:relative_path", "l8:date", "l8:cloud_cover"]].to_string(index=False))

### 4.3 Imagery

In [ ]:
tile_idx = 120

fig, axes = plt.subplots(2, 6, figsize=(24, 8))
for i in range(6):
    with rasterio.open(f"/vsicurl/{BASE_URL}/burst/DATA/{tile_idx}/s2/img_{i:02d}.tif") as src:
        axes[0, i].imshow(rgb(src))
    with rasterio.open(f"/vsicurl/{BASE_URL}/burst/DATA/{tile_idx}/l8/img_{i:02d}.tif") as src:
        axes[1, i].imshow(rgb(src))
    axes[0, i].set_title(f"S2 img_{i:02d}", fontsize=10)
    axes[1, i].set_title(f"L8 img_{i:02d}", fontsize=10)
for ax in axes.flat:
    ax.axis("off")
plt.suptitle(f"Burst, tile {tile_idx}", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Added layers: Sentinel-1, cloud masks, OpenStreetMap

Every S2 frame has a Sentinel-1 RTC image (VV, VH, linear gamma-0) acquired as close as possible to it, on the same grid, and every S2 and Landsat frame has a cloud and shadow mask (0 clear, 1 thick cloud, 2 thin cloud, 3 shadow). A frame with no Sentinel-1 pass holds no valid pixels; `children__s1.parquet` gives each image's `s1:valid_fraction`.

In [ ]:
tile_idx = 120
df_c_s1 = get_parquet("burst/METADATA/children__s1.parquet")
print(frame_rows(df_c_s1, tile_idx)[["internal:relative_path", "s1:s2_date", "s1:datetime",
                                      "s1:orbit_state", "s1:valid_fraction"]].to_string(index=False))

mask_cmap = plt.matplotlib.colors.ListedColormap(["#2a9d8f", "#f4f1de", "#a8dadc", "#3d405b"])
fig, axes = plt.subplots(3, 6, figsize=(24, 12))
for i in range(6):
    with rasterio.open(f"/vsicurl/{BASE_URL}/burst/DATA/{tile_idx}/s2/img_{i:02d}.tif") as src:
        axes[0, i].imshow(rgb(src))
    mask = np.array(Image.open(io.BytesIO(requests.get(
        f"{BASE_URL}/burst/DATA/{tile_idx}/cloud_s2/img_{i:02d}.png", headers=HEADERS).content)))
    axes[1, i].imshow(mask, cmap=mask_cmap, vmin=0, vmax=3, interpolation="nearest")
    with rasterio.open(f"/vsicurl/{BASE_URL}/burst/DATA/{tile_idx}/s1/img_{i:02d}.tif") as src:
        vv = src.read(1)
    axes[2, i].imshow(10 * np.log10(np.where(vv > 0, vv, np.nan)), cmap="gray", vmin=-25, vmax=0)
    axes[0, i].set_title(f"S2 img_{i:02d}")
    axes[1, i].set_title("cloud mask")
    axes[2, i].set_title("S1 VV (dB)")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

`osm.parquet` and `admin.parquet` are GeoParquet files in the tile's UTM projection: every OpenStreetMap feature clipped to the tile, and the administrative units it intersects (© OpenStreetMap contributors, ODbL-1.0).

In [ ]:
osm = get_geoparquet(f"burst/DATA/{tile_idx}/osm.parquet")
admin = get_geoparquet(f"burst/DATA/{tile_idx}/admin.parquet")
print(admin[["name", "admin_level"]].to_string(index=False))
print(osm["category"].value_counts().head(10))

fig, ax = plt.subplots(figsize=(8, 8))
osm.plot(ax=ax, column="category", legend=True, linewidth=0.8, markersize=4,
         legend_kwds={"loc": "upper left", "bbox_to_anchor": (1, 1)})
ax.set_title(f"OpenStreetMap, burst tile {tile_idx}")
ax.set_axis_off()
plt.show()

## 6. PyTorch DataLoader

A minimal DataLoader that streams tiles directly from Source Cooperative. No downloads needed, `rasterio` reads remote COGs via HTTP range requests. The example below uses Part A (monotemporal). The same pattern extends to B and C by adjusting the path template for the series leaves (`img_{i:02d}.tif` instead of `data.tif`).

In [ ]:
%pip install -q obstore torch

In [ ]:
import torch
from concurrent.futures import ThreadPoolExecutor
from obstore.store import HTTPStore
from rasterio.io import MemoryFile
from torch.utils.data import DataLoader, Dataset


class ElliotMonotemporal(Dataset):
    """Streaming dataset with batched parallel fetching.

    Uses __getitems__ so the DataLoader fetches an entire batch in
    parallel via obstore plus ThreadPoolExecutor instead of sequential
    __getitem__ calls. obstore releases the GIL, so threads run truly
    in parallel.
    """

    def __init__(self, df, base_url, modalities=("s2", "l8", "dem", "lc", "s1")):
        self.paths = df["internal:relative_path"].tolist()
        self.modalities = modalities
        self.store = HTTPStore.from_url(base_url)

    def __len__(self):
        return len(self.paths)

    def __getitems__(self, indices):
        jobs = [(idx, mod, f"DATA/{self.paths[idx]}/{mod}/data.tif")
                for idx in indices for mod in self.modalities]
        with ThreadPoolExecutor(max_workers=len(jobs)) as pool:
            buffers = list(pool.map(lambda j: bytes(self.store.get(j[2]).bytes()), jobs))
        samples = {i: {"tile_id": self.paths[i]} for i in indices}
        for (idx, mod, _), buf in zip(jobs, buffers):
            samples[idx][mod] = self._decode(buf)
        return [samples[i] for i in indices]

    @staticmethod
    def _decode(buf):
        with MemoryFile(buf) as mem:
            with mem.open() as src:
                return torch.from_numpy(src.read().astype(np.float32))


def collate_fn(batch):
    """Stack each modality separately since they have different shapes."""
    collated = {"tile_id": [s["tile_id"] for s in batch]}
    for mod in [k for k in batch[0] if k != "tile_id"]:
        collated[mod] = torch.stack([s[mod] for s in batch])
    return collated

In [ ]:
ds = ElliotMonotemporal(spain.iloc[:100], f"{BASE_URL}/monotemporal", modalities=("s2", "s1", "dem"))
loader = DataLoader(ds, batch_size=16, collate_fn=collate_fn)

In [ ]:
%%time
batch = next(iter(loader))
print(batch["s2"].shape)   # [16, 13, 1056, 1056]
print(batch["s1"].shape)   # [16,  2, 1056, 1056]
print(batch["dem"].shape)  # [16,  1, 1056, 1056]

For typed reading of every layer (bands, units, scaling, frame times) see `example_taco.ipynb`, which reads the same data through the TACO reader.